In [72]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize

In [73]:
a = 0.0
start = 2
x_grid = np.linspace(-2, 2, 500)
def potential(x):
    return (x[0]**2 - 1)**2 + a*x[0]

def gradient(x):
    return np.array([
        4*x[0]*(x[0]**2 - 1) + a
    ])

In [74]:
def bfgs(f, grad, x0, tol=1e-8, max_iter=5000):
    return minimize(
        f, x0, jac=grad, method="BFGS",
        options={"gtol": tol, "maxiter": max_iter, "return_all": True}
    )

In [ ]:
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

bfgs_result = bfgs(potential, gradient, start)
if not bfgs_result.success:
    raise RuntimeError(f"BFGS did not converge: {bfgs_result.message}")
current_history = np.asarray(bfgs_result.allvecs)[:, 0]
best_x = current_history[0]
best_energy = float(potential([best_x]))
best_history = []

for current_x in current_history:
    current_energy = float(potential([current_x]))
    if current_energy < best_energy:
        best_x = current_x
        best_energy = current_energy
    best_history.append(best_x)

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(x_grid, [potential([xi]) for xi in x_grid], color="black", linewidth=2)
current_point, = ax.plot([], [], "o", color="red", markersize=12)
best_point, = ax.plot([], [], "o", markeredgecolor="black", markerfacecolor="none", markersize=14)
ax.set(xlim=(-2, 2), xlabel="x", ylabel="V(x)", title="BFGS")


def update(frame):
    current = current_history[frame]
    best = best_history[frame]
    current_point.set_data([current], [potential([current])])
    best_point.set_data([best], [potential([best])])
    return current_point, best_point

animation = FuncAnimation(
    fig, update, frames=len(current_history), interval=100, blit=True, repeat=False
)
plt.close(fig)
HTML(animation.to_jshtml())